In [ ]:
# Cell 1 — Setup and inputs.
#
# Loads the Python libraries used across every subsequent cell (geopandas,
# pyproj, requests, etc.), declares the building input as either an EGID
# (works for 17 EGID-indexed cantons: ZH, BE, ...) or a WGS84 coordinate
# (works anywhere, including non-EGID cantons like VD where the hackathon
# is held, and ZG), creates the tile cache directory under data/ (already
# gitignored via data/), and loads the Google Street View API key from .env
# for later cells.
#
# No network access and no data reads happen here — this is pure
# configuration, the "#define block" of the notebook. Both input options
# should succeed regardless of values because nothing is looked up yet.

from __future__ import annotations

import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

# Option A: EGID-indexed canton (ZH, BE, ...). 17 cantons supported.
EGID: str | None = "140305"  # Zurich central, known multi-view problem case
LAT: float | None = None
LON: float | None = None

# Option B: coordinate fallback for non-EGID cantons (VD = hackathon venue, ZG).
# Uncomment these and set EGID = None to test the spatial-lookup path.
# EGID = None
# LAT = 46.5197  # central Lausanne (near Place St-Francois)
# LON = 6.6335

REPO_ROOT = Path("/Users/lukemarinos/WORK/glassscan")
DATA_DIR = REPO_ROOT / "data"
TILE_DIR = DATA_DIR / "swissbuildings3d" / "tiles"
TILE_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(REPO_ROOT / ".env")
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]

STAC_COLLECTION_URL = (
    "https://data.geo.admin.ch/api/stac/v0.9/collections/"
    "ch.swisstopo.swissbuildings3d_3_0"
)

print(f"Tile cache: {TILE_DIR}")
print(f"EGID={EGID}  LAT={LAT}  LON={LON}")


In [ ]:
# Cell 2 — Inspect the STAC collection metadata.
#
# Makes one HTTPS GET to the swisstopo catalog endpoint. No authentication
# is needed — swisstopo's STAC is fully open. The response is a ~5 KB JSON
# document describing the whole collection (swissBUILDINGS3D 3.0), not
# individual tiles: its spatial extent (all of Switzerland), temporal range
# (when the source data was captured), license, and the hypermedia links
# we follow to reach tile-level Items.
#
# Think of it as the index page of a library rather than any specific book.
# We do this as a cheap contract check: if swisstopo migrates to STAC v1 or
# renames endpoints, this cell fails early and loudly, before we build
# downstream code that silently breaks. The key output to pay attention to
# is the `rel: items` link in the Links block — that's the URL cell 3 hits
# to walk the tiles.

r = requests.get(STAC_COLLECTION_URL, timeout=30)
r.raise_for_status()
collection = r.json()

print("Top-level keys:", list(collection.keys()))
print()
print(f"ID:    {collection.get('id')}")
print(f"Title: {collection.get('title')}")
print()
print("Description (first 300 chars):")
print(collection.get("description", "")[:300])
print()

extent = collection.get("extent", {})
spatial_bbox = extent.get("spatial", {}).get("bbox")
temporal = extent.get("temporal", {}).get("interval")
print(f"Spatial extent (bbox): {spatial_bbox}")
print(f"Temporal extent:       {temporal}")
print()

print("Links:")
for link in collection.get("links", []):
    print(f"  {link.get('rel'):<15} {link.get('href', '')}")


In [ ]:
# Cell 3 — Walk the items endpoint (first page, no filter).
#
# Each STAC Item represents one tile — a ~1-6 km square chunk of Switzerland
# with its own WGS84 bbox, capture date, and downloadable assets (a
# .gdb.zip for ESRI and a .dwg.zip for AutoCAD). There are roughly a few
# thousand tiles covering the country.
#
# We fetch page 1 unfiltered to learn the Item structure before narrowing
# down: what does a tile id look like, what MIME types does each carry,
# is the geometry a Polygon, what's in properties. Once we know, cell 5
# will send the SAME endpoint a `?bbox=` query and get back only the
# tile(s) covering our building — so we never actually need to walk all
# pages via the `rel: next` cursor. This cell is purely reconnaissance.

items_url = f"{STAC_COLLECTION_URL}/items"
r = requests.get(items_url, timeout=30)
r.raise_for_status()
items = r.json()

print("Top-level keys:", list(items.keys()))
print(f"Features on this page: {len(items.get('features', []))}")
if "numberMatched" in items:
    print(f"Total tiles in collection: {items['numberMatched']}")
print()

# Pagination links
print("Page links:")
for link in items.get("links", []):
    if link.get("rel") in ("self", "next", "prev", "first"):
        print(f"  {link.get('rel'):<6} {link.get('href', '')}")
print()

# Examine one tile Item in detail
if items.get("features"):
    tile = items["features"][0]
    print("One tile Item:")
    print(f"  id:       {tile.get('id')}")
    print(f"  type:     {tile.get('type')}")
    print(f"  bbox:     {tile.get('bbox')}")
    print(f"  geometry: {type(tile.get('geometry')).__name__}")
    props = tile.get("properties", {})
    print(f"  datetime: {props.get('datetime')}")
    print(f"  assets:")
    for name, asset in tile.get("assets", {}).items():
        href = asset.get("href", "")
        ctype = asset.get("type", "?")
        print(f"    {name}: {href[:90]}")
        print(f"      type={ctype}")


In [ ]:
# Cell 4 — Resolve the input to a canonical pair of coordinates.
#
# Two branches depending on which Option was set in cell 1. If EGID is
# given, we hit the GWR API (api3.geo.admin.ch) which returns the
# building's point location in WGS84 along with its registered attributes:
# address (strname_deinr), construction year (gbauj), storey count (gastw),
# floor area in m2 (garea), volume in m3 (gvol), and category code (gkat).
# These attributes double as features for the predict module and as a
# sanity check later against whatever swissBUILDINGS3D gives us. If
# coordinates were given instead, we skip the API call and pass them
# through.
#
# In both cases we then reproject WGS84 -> LV95 with pyproj and keep BOTH
# forms. WGS84 is the outside-world language (Google Street View, STAC
# bbox filters, folium basemaps). LV95 is the inside-world language (GDB
# polygons, distance-in-metres math, point-in-polygon against building
# footprints). Reprojecting the single query point once here is cheap; the
# alternative is reprojecting thousands of polygon vertices on the fly
# later, which is both slower and a common source of axis-order bugs.
#
# The `always_xy=True` flag on each Transformer forces (x, y) = (lon, lat)
# for WGS84 and (x, y) = (E, N) for LV95 everywhere. Without it, pyproj
# uses authority-defined axis order which differs per CRS (WGS84 puts lat
# first, LV95 puts E first). Mixing those silently plants buildings in
# the wrong canton; always pin it.

from pyproj import Transformer

to_lv95 = Transformer.from_crs("EPSG:4326", "EPSG:2056", always_xy=True)
to_wgs = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True)

GWR_FIND_URL = "https://api3.geo.admin.ch/rest/services/api/MapServer/find"

gwr_attrs: dict = {}
if EGID is not None:
    params = {
        "layer": "ch.bfs.gebaeude_wohnungs_register",
        "searchText": EGID,
        "searchField": "egid",
        "returnGeometry": "true",
        "sr": "4326",
    }
    r = requests.get(GWR_FIND_URL, params=params, timeout=30)
    r.raise_for_status()
    results = r.json().get("results", [])
    if not results:
        raise RuntimeError(f"No GWR entry found for EGID={EGID}")
    hit = results[0]
    gwr_attrs = hit.get("attributes", {})
    geom = hit.get("geometry", {})
    LON_WGS = geom.get("x")
    LAT_WGS = geom.get("y")
    print(f"GWR returned {len(results)} result(s) for EGID={EGID}; using first.")
    print(f"  address:   {gwr_attrs.get('strname_deinr')}")
    print(f"  built:     {gwr_attrs.get('gbauj')}")
    print(f"  storeys:   {gwr_attrs.get('gastw')}")
    print(f"  area_m2:   {gwr_attrs.get('garea')}")
    print(f"  volume_m3: {gwr_attrs.get('gvol')}")
    print(f"  category:  {gwr_attrs.get('gkat')}")
elif LAT is not None and LON is not None:
    LAT_WGS, LON_WGS = LAT, LON
    print(f"Using provided coordinates (no GWR lookup).")
else:
    raise ValueError("Set EGID or both LAT and LON in cell 1")

E_LV95, N_LV95 = to_lv95.transform(LON_WGS, LAT_WGS)

print()
print(f"WGS84:  lat={LAT_WGS:.6f}  lon={LON_WGS:.6f}")
print(f"LV95:   E={E_LV95:.2f}  N={N_LV95:.2f}")


In [9]:
# Cell 5 — Find the local tile that covers our building.
#
# Builds a tiny bbox around the WGS84 point (+/- 0.0005 deg, about +/- 50 m)
# and sends it to the STAC items endpoint as
# `?bbox=minlon,minlat,maxlon,maxlat`. The server returns only tile Items
# whose own bbox intersects our query box.
#
# Important finding: swisstopo publishes TWO kinds of Items in this
# collection.
#   1. Regional tiles, e.g. `swissbuildings3d_3_0_2020_1243-14`, each
#      covering a ~6x3 km chunk of Switzerland. File size ~10-20 MB. This
#      is what we want.
#   2. Annual country-wide snapshots, e.g. `swissbuildings3d_3_0_2025`, each
#      a single GDB containing all of Switzerland. File size ~55 GB.
# Any point-bbox query near Switzerland returns both a regional tile AND
# all annual snapshots (because their bboxes span the whole country). We
# must explicitly filter them out or we risk picking a 55 GB download.
# The filter is a bbox-span test: regional tiles are at most ~1 deg wide,
# country-wide items are ~6 deg wide, so any threshold between catches it.
#
# Among the remaining local tiles we pick the one with the most recent
# capture date — swisstopo sometimes has multiple generations of the same
# region (e.g. a 2013 flyover and a 2020 re-flyover) and we want the
# freshest building geometry. The picked download URL is stored as
# GDB_URL for cell 6.
#
# We also pick on MIME type `application/x.filegdb+zip` rather than file
# extension — robust to URL-shape changes — and skip the .dwg.zip AutoCAD
# variant which is painful to read from Python.

eps = 0.0005  # ~50 m; safe for tile-boundary edge cases
bbox = [LON_WGS - eps, LAT_WGS - eps, LON_WGS + eps, LAT_WGS + eps]

params = {"bbox": ",".join(f"{v:.6f}" for v in bbox)}
r = requests.get(f"{STAC_COLLECTION_URL}/items", params=params, timeout=30)
r.raise_for_status()
matches = r.json().get("features", [])

print(f"Query bbox: {params['bbox']}")
print(f"Total matches: {len(matches)}")
print()


def _bbox_span_deg(b: list[float]) -> float:
    """Max side length of a WGS84 bbox in degrees."""
    return max(b[2] - b[0], b[3] - b[1])


# Threshold: regional tiles are at most ~0.1 deg wide; country-wide items
# are ~6 deg wide. 1.0 is comfortably between.
LOCAL_TILE_MAX_SPAN_DEG = 1.0

# Summarise every match, tagging each as country-wide or local.
local_tiles: list[dict] = []
for i, tile in enumerate(matches):
    tid = tile.get("id")
    tbbox = tile.get("bbox", [])
    tdate = tile.get("properties", {}).get("datetime")
    span = _bbox_span_deg(tbbox) if tbbox else float("inf")
    kind = "COUNTRY-WIDE (skipped)" if span > LOCAL_TILE_MAX_SPAN_DEG else "local"
    print(f"[{i}] {tid}  [{kind}]")
    print(f"    bbox span: {span:.3f} deg")
    print(f"    captured:  {tdate}")
    if span <= LOCAL_TILE_MAX_SPAN_DEG:
        local_tiles.append(tile)
print()

if not local_tiles:
    raise RuntimeError(
        "No local tile covers this point — only country-wide snapshots matched."
    )

# Pick the most recently captured local tile.
local_tiles.sort(
    key=lambda t: t.get("properties", {}).get("datetime") or "",
    reverse=True,
)
picked = local_tiles[0]

GDB_URL: str | None = None
TILE_ID = picked.get("id")
TILE_DATETIME = picked.get("properties", {}).get("datetime")

for name, asset in picked.get("assets", {}).items():
    if asset.get("type") == "application/x.filegdb+zip":
        GDB_URL = asset.get("href")
        break

if GDB_URL is None:
    raise RuntimeError("Selected tile has no .gdb.zip asset.")

print(f"Selected tile: {TILE_ID}")
print(f"Captured:      {TILE_DATETIME}")
print(f"GDB URL:       {GDB_URL}")


Query bbox: 6.633000,46.519200,6.634000,46.520200
Total matches: 4

[0] swissbuildings3d_3_0_2020_1243-14  [local]
    bbox span: 0.057 deg
    captured:  2020-01-01T00:00:00Z
[1] swissbuildings3d_3_0_2023  [COUNTRY-WIDE (skipped)]
    bbox span: 6.035 deg
    captured:  2023-11-29T00:00:00Z
[2] swissbuildings3d_3_0_2024  [COUNTRY-WIDE (skipped)]
    bbox span: 6.035 deg
    captured:  2024-11-19T00:00:00Z
[3] swissbuildings3d_3_0_2025  [COUNTRY-WIDE (skipped)]
    bbox span: 6.035 deg
    captured:  2025-11-26T00:00:00Z

Selected tile: swissbuildings3d_3_0_2020_1243-14
Captured:      2020-01-01T00:00:00Z
GDB URL:       https://data.geo.admin.ch/ch.swisstopo.swissbuildings3d_3_0/swissbuildings3d_3_0_2020_1243-14/swissbuildings3d_3_0_2020_1243-14_2056_5728.gdb.zip


In [12]:
# Cell 6 — Download the selected tile's .gdb.zip to the local cache.
#
# Before the actual GET we fire an HTTP HEAD request to get Content-Length.
# This is a safety belt on top of cell 5's country-wide filter: if the URL
# unexpectedly points at a >500 MB file, we abort rather than saturate the
# connection. Any legitimate regional tile is ~10-20 MB, so 500 MB is a
# comfortably loose cap that only catches "something went very wrong".
#
# The download itself uses streaming (stream=True + iter_content) so the
# whole file isn't held in memory at once. Overkill for 20 MB but cheap
# and correct.
#
# We cache by tile id: if `TILE_DIR/<TILE_ID>.gdb.zip` already exists we
# skip the download entirely. This makes the cell idempotent — rerunning
# the whole notebook doesn't re-hit swisstopo every time. Delete the file
# from disk if you ever want to force a fresh pull (e.g. to test a newer
# tile generation).
#
# At the end we have a .gdb.zip on disk. Cell 7 will unzip it, inspect
# what layers live inside (the "separated elements" vs "closed solids"
# variants mentioned in the collection description should become visible
# here) and open them with geopandas.

import shutil

MAX_DOWNLOAD_MB = 500  # safety cap; regional tiles are ~10-20 MB

TILE_ZIP_PATH = TILE_DIR / f"{TILE_ID}.gdb.zip"

if TILE_ZIP_PATH.exists():
    size_mb = TILE_ZIP_PATH.stat().st_size / (1024 * 1024)
    print(f"Already cached: {TILE_ZIP_PATH}")
    print(f"Size:           {size_mb:.1f} MB")
else:
    # HEAD: sanity-check size before committing to the GET
    h = requests.head(GDB_URL, timeout=30, allow_redirects=True)
    h.raise_for_status()
    content_length = int(h.headers.get("Content-Length", 0))
    expected_mb = content_length / (1024 * 1024)
    print(f"HEAD Content-Length: {expected_mb:.1f} MB")

    if expected_mb > MAX_DOWNLOAD_MB:
        raise RuntimeError(
            f"Tile is {expected_mb:.0f} MB, exceeds safety cap "
            f"({MAX_DOWNLOAD_MB} MB). Likely a country-wide snapshot slipped "
            "through cell 5's filter. Investigate before downloading."
        )

    print(f"Downloading to {TILE_ZIP_PATH} ...")
    with requests.get(GDB_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(TILE_ZIP_PATH, "wb") as f:
            shutil.copyfileobj(r.raw, f, length=1024 * 1024)  # 1 MB chunks

    actual_mb = TILE_ZIP_PATH.stat().st_size / (1024 * 1024)
    print(f"Downloaded:     {actual_mb:.1f} MB")

print()
print(f"Tile zip path:  {TILE_ZIP_PATH}")


Already cached: /Users/lukemarinos/WORK/glassscan/data/swissbuildings3d/tiles/swissbuildings3d_3_0_2020_1243-14.gdb.zip
Size:           45.3 MB

Tile zip path:  /Users/lukemarinos/WORK/glassscan/data/swissbuildings3d/tiles/swissbuildings3d_3_0_2020_1243-14.gdb.zip


In [17]:
# Cell 7 — Extract the .gdb.zip and inspect its layers.
#
# A File Geodatabase (.gdb) is a folder, not a single file. When you
# unzip `<tile>.gdb.zip` you get a folder `<tile>.gdb/` containing
# dozens of .gdbtable / .gdbindexes binary files. Inside are multiple
# LAYERS (think: tables), each holding a different type of feature:
# roofs, walls, floors, closed building solids, etc.
#
# Parsing this GDB is surprisingly fiddly. Two failed approaches
# preceded the one used here:
#
#   1. `fiona.open(path, layer=...)` then `src.schema` — raises
#      `UnsupportedGeometryTypeError: 2147483648` (GDAL code
#      0x80000000, used internally for ESRI MultiPatch). fiona's
#      schema normaliser can't map that to a supported WKB type.
#   2. `pyogrio.list_layers` / `pyogrio.read_info` / even
#      `pyogrio.read_dataframe(read_geometry=False)` — same error
#      from pyogrio's own geometry-type check. Both Python libraries
#      probe the layer's declared geometry type at open time and
#      both choke on the MultiPatch code.
#
# The workaround: skip .schema and iterate features directly. fiona
# can iterate features fine because each feature's own WKB decodes
# to a standard MultiPolygon (the issue is only with the layer-level
# declared type, not with the actual geometries). `fiona.listlayers()`,
# `len(src)`, `src.bounds`, and `src.crs` also all work without
# touching `.schema`. We use these to report per-layer metadata, then
# pull fields and geometry type from the first feature instead of the
# schema.
#
# Expected layers based on the collection description:
#   - Separated elements: Floor, Wall, Roof — per-surface polygons
#     for each building (useful for facade extraction).
#   - Closed solids: Building_solid, Roof_solid — whole buildings
#     as single 3D shells.
# Key fields to look for: EGID (for Option A filter), OBJEKTART
# (building-element type), GESAMTHOEHE (total height), DACH_MIN /
# DACH_MAX (roof lower/upper edge elevations).

import zipfile
import fiona

# Extract if not already done. Any .gdb folder already under TILE_DIR
# implies a prior run extracted something; skip to stay idempotent.
existing = list(TILE_DIR.glob("**/*.gdb"))
if not existing:
    print(f"Extracting {TILE_ZIP_PATH.name} ...")
    with zipfile.ZipFile(TILE_ZIP_PATH) as z:
        z.extractall(TILE_DIR)
    print("Done.")
else:
    print(f"Already extracted ({len(existing)} .gdb folder(s) present).")
print()

# Pick the .gdb folder for this tile. swisstopo's filename uses a
# slightly different case/hyphenation than the STAC id, so the
# substring match may miss; fall back to the sole folder.
all_gdbs = sorted(TILE_DIR.glob("**/*.gdb"))
candidates = [p for p in all_gdbs if TILE_ID in p.name]
if candidates:
    GDB_PATH = candidates[0]
elif len(all_gdbs) == 1:
    GDB_PATH = all_gdbs[0]
else:
    raise RuntimeError(
        f"Could not identify GDB for {TILE_ID} among {[p.name for p in all_gdbs]}"
    )
print(f"GDB: {GDB_PATH}")
print()

# fiona.listlayers works because it doesn't parse per-layer schema.
layer_names = fiona.listlayers(str(GDB_PATH))
print(f"Layers ({len(layer_names)}):")
print()

for name in layer_names:
    with fiona.open(str(GDB_PATH), layer=name) as src:
        # These all work without triggering schema parsing.
        n = len(src)
        bounds = src.bounds
        crs = str(src.crs)
        # Touching .schema would raise; pull fields/geom type from first feature.
        first_feat = next(iter(src), None)
        if first_feat is not None:
            fields = list(first_feat["properties"].keys())
            geom = first_feat.get("geometry")
            geom_type = geom["type"] if geom else None
        else:
            fields = []
            geom_type = None
    print(f"  - {name}")
    print(f"      geometry: {geom_type}")
    print(f"      features: {n}")
    print(f"      bounds:   ({bounds[0]:.1f}, {bounds[1]:.1f}) -> ({bounds[2]:.1f}, {bounds[3]:.1f})  [{crs}]")
    print(f"      fields:   {fields}")
    print()


Already extracted (1 .gdb folder(s) present).

GDB: /Users/lukemarinos/WORK/glassscan/data/swissbuildings3d/tiles/swissBUILDINGS3D_3-0_1243-14.gdb

Layers (5):

  - Floor
      geometry: MultiPolygon
      features: 5315
      bounds:   (2536814.1, 1151968.0) -> (2541257.2, 1155026.1)  [EPSG:2056]
      fields:   ['UUID', 'OBJEKTART', 'NAME_KOMPLETT', 'GEBAEUDE_NUTZUNG', 'GRUND_AENDERUNG', 'HERKUNFT', 'HERKUNFT_MONAT', 'HERKUNFT_JAHR', 'ORIGINAL_HERKUNFT', 'ERSTELLUNG_MONAT', 'ERSTELLUNG_JAHR', 'REVISION_JAHR', 'REVISION_MONAT', 'DATUM_ERSTELLUNG', 'DATUM_AENDERUNG', 'GELAENDEPUNKT', 'GESAMTHOEHE', 'EGID']

  - Roof_solid
      geometry: MultiPolygon
      features: 5391
      bounds:   (2536814.1, 1151968.0) -> (2541257.1, 1155026.1)  [EPSG:2056]
      fields:   ['UUID', 'OBJEKTART', 'NAME_KOMPLETT', 'GEBAEUDE_NUTZUNG', 'GRUND_AENDERUNG', 'HERKUNFT', 'HERKUNFT_JAHR', 'HERKUNFT_MONAT', 'ORIGINAL_HERKUNFT', 'ERSTELLUNG_JAHR', 'ERSTELLUNG_MONAT', 'REVISION_JAHR', 'REVISION_MONAT', 'DATUM

In [18]:
# Cell 8 — Find our target building inside the GDB.
#
# We query the `Floor` layer because it holds the 2D building
# footprints — the ground-level polygon is the most useful shape for
# downstream rectification, adaptive FOV, and neighbor masking. The
# Wall and Roof layers hold 3D surfaces and are consulted later for
# height and facade extraction.
#
# Two lookup paths, mirroring cell 1's Option A / Option B:
#
#   Option A (EGID is set): we ask fiona to attribute-filter with
#     `src.filter(where="EGID = <egid>")`. GDAL translates this to an
#     OGR SQL expression and uses an index if present; worst case it
#     sequentially scans the ~5000 features in under a second. We
#     still double-check the match in Python in case the where clause
#     returns `None` (driver-dependent).
#
#   Option B (coords only): we want the building that CONTAINS our
#     point. We first narrow candidates with a bbox filter in LV95
#     (~30 m around the query point) — this lets GDAL skip almost all
#     features cheaply — then call shapely's `.contains()` on each
#     remaining polygon for the exact hit. If the point lands in open
#     space (a square, street, park) no polygon contains it and we
#     raise with a helpful message; nudge the coords in cell 1 and
#     re-run.
#
# Output: we save the footprint (LV95), the full attribute record,
# and report a summary. The footprint is a shapely Polygon or
# MultiPolygon we can feed to later cells for projection, centroid,
# neighbor masking, etc.

from shapely.geometry import Point, shape

target_egid: str | None = None
target_footprint = None
target_attrs: dict = {}

with fiona.open(str(GDB_PATH), layer="Floor") as src:
    if EGID is not None:
        # Option A: attribute filter. Fall back to scan if `where` unsupported.
        try:
            iter_feats = src.filter(where=f"EGID = {EGID}")
        except Exception:
            iter_feats = iter(src)
        for feat in iter_feats:
            props = feat["properties"]
            if str(props.get("EGID")) == str(EGID):
                target_egid = str(EGID)
                target_footprint = shape(feat["geometry"])
                target_attrs = dict(props)
                break
    else:
        # Option B: bbox pre-filter + shapely contains.
        point_lv95 = Point(E_LV95, N_LV95)
        eps_m = 30
        for feat in src.filter(bbox=(
            E_LV95 - eps_m, N_LV95 - eps_m,
            E_LV95 + eps_m, N_LV95 + eps_m,
        )):
            poly = shape(feat["geometry"])
            if poly.contains(point_lv95):
                target_egid = str(feat["properties"].get("EGID"))
                target_footprint = poly
                target_attrs = dict(feat["properties"])
                break

if target_footprint is None:
    msg = (
        f"No Floor polygon matches EGID={EGID}."
        if EGID is not None
        else f"No Floor polygon contains point ({E_LV95:.2f}, {N_LV95:.2f}). "
        f"Query coord may be in open space — nudge LAT/LON in cell 1."
    )
    raise RuntimeError(msg)

# Centroid in both CRS — useful as a canonical "aim at this building" target
# for Street View. Often a better camera target than the input coord, which
# might sit near the building's edge.
centroid_lv95 = target_footprint.centroid
CENTROID_LON, CENTROID_LAT = to_wgs.transform(centroid_lv95.x, centroid_lv95.y)

print(f"Found building: EGID={target_egid}")
print(f"  OBJEKTART:        {target_attrs.get('OBJEKTART')}")
print(f"  GESAMTHOEHE:      {target_attrs.get('GESAMTHOEHE')} m")
print(f"  GEBAEUDE_NUTZUNG: {target_attrs.get('GEBAEUDE_NUTZUNG')}")
print(f"  HERKUNFT:         {target_attrs.get('HERKUNFT')} ({target_attrs.get('HERKUNFT_JAHR')})")
print()
print("Footprint:")
print(f"  geom_type:     {target_footprint.geom_type}")
print(f"  bounds (LV95): {tuple(round(v, 1) for v in target_footprint.bounds)}")
print(f"  area:          {target_footprint.area:.1f} m^2")
print(f"  centroid LV95: ({centroid_lv95.x:.2f}, {centroid_lv95.y:.2f})")
print(f"  centroid WGS:  ({CENTROID_LAT:.6f}, {CENTROID_LON:.6f})")


Found building: EGID=None
  OBJEKTART:        Sakrales Gebaeude
  GESAMTHOEHE:      None m
  GEBAEUDE_NUTZUNG: None
  HERKUNFT:         swisstopo (2012)

Footprint:
  geom_type:     MultiPolygon
  bounds (LV95): (2538178.3, 1152345.0, 2538232.6, 1152376.1)
  area:          1176.7 m^2
  centroid LV95: (2538203.79, 1152361.20)
  centroid WGS:  (46.519690, 6.633334)


In [ ]:
# Cell 9 — Visualise the footprint on a real basemap.
#
# Pure sanity check. After cell 8 we have a polygon expressed as LV95
# metres — abstract numbers we can't eyeball. This cell reprojects the
# polygon to WGS84 and drops it on a Leaflet map (via folium) so we
# can confirm visually that we picked the right building.
#
# What the map shows:
#   - red polygon: the matched footprint (target_footprint).
#   - red marker:  the footprint centroid (CENTROID_LAT, CENTROID_LON).
#                  This is the "aim at this building" target we'll hand
#                  to Street View later.
#   - blue marker: your input point (LAT_WGS, LON_WGS).
#                  In Option B this is whatever coord you typed in cell 1;
#                  in Option A it's the GWR-resolved point. The two
#                  markers should sit near each other and inside the
#                  polygon. If they're far apart or outside it, cell 8
#                  matched the wrong building and we should investigate.
#
# Two basemap layers are wired up so you can toggle in the top-right:
#   - OpenStreetMap (default): clearer street labels.
#   - swissimage (aerial photo from swisstopo): best for confirming the
#     building shape against reality.
#
# Reprojection uses shapely.ops.transform with the pyproj `to_wgs`
# Transformer set up in cell 4. Folium's GeoJson layer wants WGS84
# (lon, lat) tuples, which `always_xy=True` already gives us.

import folium
from shapely.ops import transform as shp_transform

footprint_wgs = shp_transform(
    lambda x, y, z=None: to_wgs.transform(x, y),
    target_footprint,
)

m = folium.Map(
    location=[CENTROID_LAT, CENTROID_LON],
    zoom_start=19,
    tiles="OpenStreetMap",
)

# Swiss aerial imagery — better than OSM for confirming building shapes.
folium.TileLayer(
    tiles="https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.swissimage/default/current/3857/{z}/{x}/{y}.jpeg",
    attr="\u00a9 swisstopo",
    name="swissimage (aerial)",
    overlay=False,
).add_to(m)

folium.GeoJson(
    footprint_wgs.__geo_interface__,
    name="target footprint",
    style_function=lambda x: {
        "color": "#d33",
        "weight": 2,
        "fillColor": "#d33",
        "fillOpacity": 0.3,
    },
).add_to(m)

folium.Marker(
    [CENTROID_LAT, CENTROID_LON],
    popup=f"Centroid<br>EGID={target_egid}",
    icon=folium.Icon(color="red"),
).add_to(m)

folium.Marker(
    [LAT_WGS, LON_WGS],
    popup="Input point",
    icon=folium.Icon(color="blue"),
).add_to(m)

folium.LayerControl().add_to(m)

m
